In [ ]:
#loading/preparing data

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import numpy as np
from sklearn.metrics import mean_squared_error, confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import SGD
from scikeras.wrappers import KerasClassifier

#load/prep data
df = pd.read_csv('final_project_data.csv')

#get cols
x_cols = df.columns[:-1]
y_col = df.columns[-1]

#get data
x = df[x_cols].values.astype(np.float64)
y = df[y_col].values.astype(int)

#split into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, 
random_state=42)

#save some for validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, 
random_state=42)


In [ ]:
#setting up MLP

#make MLP
def build_clf(hidden1 = 32, rate = 0.01, dropout_rate = 0, **kwargs):
    model = Sequential([
        Input(shape=(x.shape[1],)),
        Dense(hidden1, activation = 'relu'),
        Dropout(dropout_rate),
        Dense(1, activation = 'sigmoid')
    ])

#https://www.tensorflow.org/api_docs/python/tf/keras/Model
    
    model.compile(optimizer=SGD(learning_rate = rate), loss = 'binary_crossentropy',
                  metrics = ['accuracy'])
    return model

model = KerasClassifier(model=build_clf, epochs = 50, batch_size = 128, verbose = 1)

#test diff params
params = {
    'model__hidden1':[32, 64, 128],
    'model__rate':[0.0001, 0.0005, 0.001, 0.005, 0.01],
    'epochs': [50, 100, 150],
    'model__dropout_rate': [0, 0.1, 0.3, 0.5]
}
#class sklearn.model_selection.RandomizedSearchCV(estimator, param_distributions, 
#*, n_iter=10, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, 
#pre_dispatch='2*n_jobs', random_state=None, error_score=nan, return_train_score=False)

#using random search 
random_search = RandomizedSearchCV(estimator = model, param_distributions = params,
                    n_iter = 25, scoring = 'f1', cv = 3, verbose = 1,
                    random_state = 42)

In [ ]:
#fitting

# #fit on training data
original_rs = random_search.fit(x_train,y_train)

#original_rs = joblib.load("1layeroriginal.pkl")

print("Best CV F1: ", original_rs.best_score_)
print("Best Params: ", original_rs.best_params_)

In [ ]:
np.random.seed(42)

original_best_model = original_rs.best_estimator_

y_pred = original_best_model.predict(x_test)

#full confusion matrix printed below graph
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:")
print(cm)

f1 = f1_score(y_test, y_pred)

print("F1 Score: ", f1)
print("Classification")
print(classification_report(y_test,y_pred))

In [ ]:
#https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html
from sklearn.utils.class_weight import compute_class_weight
#majority of data was class 1, so model just learned to always predict 1
#had to increase weight of class 0 so it would actually predict
import random
np.random.seed(42)
random.seed(42)

classes = np.array([0,1])
weights = compute_class_weight(class_weight="balanced", classes = classes, y = y_train)

weight_class = {0: weights[0], 1: weights[1]}
print(weight_class)

In [ ]:
#fitting

#fit on training data
rs = random_search.fit(x_train,y_train, class_weight=weight_class)

print("Best CV F1: ", rs.best_score_)
print("Best Params: ", rs.best_params_)


In [ ]:
#final best model + f1 scores eval

#use test data to get f1 stats

np.random.seed(42)
random.seed(42)

best_model = rs.best_estimator_

y_pred_weighed = best_model.predict(x_test)

#full confusion matrix printed below graph
cm_1 = confusion_matrix(y_test, y_pred_weighed)

print("Confusion matrix:")
print(cm_1)

f1_1 = f1_score(y_test, y_pred_weighed)

print("F1 Score: ", f1_1)
print("Classification")
print(classification_report(y_test,y_pred_weighed))

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm_1, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Orignal 1 Hidden Layer CM')

In [ ]:
#threshold tuning for 1 layer

y_prob = best_model.predict_proba(x_val)

thresholds = np.arange(0.1, 0.5, 0.01)

best_threshold = 0.5
y_pred_best = (y_prob[:,1] > 0.5).astype(int)
best_f1 = f1_score(y_val, y_pred_best)

print(y_pred_best)
for i in thresholds :
    temp = (y_prob[:,1] > i).astype(int)
    
    score = f1_score(y_val, temp)
    if score > best_f1:
        best_f1 = score
        best_threshold = i
        y_pred_best = temp

cm_best = confusion_matrix(y_val, y_pred_best)

print("Confusion matrix:")
print(cm_best)

print(classification_report(y_val,y_pred_best))

print("F1 Score: ", best_f1)
print("Threshold: ", best_threshold)

In [ ]:
y_prob_test = best_model.predict_proba(x_test)
y_pred_test = (y_prob_test[:,1] > best_threshold).astype(int)

cm_test = confusion_matrix(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)

print("Confusion matrix:")
print(cm_test)

print(classification_report(y_test,y_pred_test))

print("F1 Score: ", f1_test)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm_test, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('0.48 Threshold 1 Hidden Layer CM')

In [ ]:



# #so we can see error of each epoch
# #history = best_model.fit(x_train, y_train, class_weight = weight_class, verbose = 0)

# y_pred_test = (test.predict(x_test)>0.33).astype(int)

# #full confusion matrix printed below graph
# cm_test = confusion_matrix(y_test, y_pred_test)

# print("Confusion matrix:")
# print(cm_test)

# f1_test = f1_score(y_test, y_pred_test)

# print("F1 Score: ", f1_test)
# print("Classification")
# print(classification_report(y_test,y_pred_test))

In [ ]:
#2 layers
#make MLP
def build_clf2(hidden1 = 32, hidden2 = 32, rate = 0.01, dropout_rate = 0, **kwargs):
    model2 = Sequential([
        Input(shape=(x.shape[1],)),
        Dense(hidden1, activation = 'relu'),
        Dense(hidden2, activation = 'relu'),
        Dropout(dropout_rate),
        Dense(1, activation = 'sigmoid')
    ])

#https://www.tensorflow.org/api_docs/python/tf/keras/Model
    
    model2.compile(optimizer=SGD(learning_rate = rate), loss = 'binary_crossentropy',
                  metrics = ['accuracy'])
    return model2

model2 = KerasClassifier(model=build_clf2, epochs = 10, batch_size = 128, verbose = 1)

#test diff params
params2 = {
    'model__hidden1':[32, 64, 128],
    'model__hidden2':[32, 64, 128],
    'model__rate':[0.0005, 0.001, 0.005],
    'epochs': [50, 100, 150],
    'model__dropout_rate': [0, 0.1, 0.3, 0.5]
}
#class sklearn.model_selection.RandomizedSearchCV(estimator, param_distributions, 
#*, n_iter=10, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, 
#pre_dispatch='2*n_jobs', random_state=None, error_score=nan, return_train_score=False)

#using random search 
random_search2 = RandomizedSearchCV(estimator = model2, param_distributions = params2,
                    n_iter = 50, scoring = 'f1', cv = 3, verbose = 1, 
                    random_state = 42)


In [ ]:
#fitting layer 2

#fit on training data
rs2 = random_search2.fit(x_train,y_train, class_weight=weight_class)

print("Best CV F1: ", rs2.best_score_)
print("Best Params: ", rs2.best_params_)


In [ ]:
#final best model + f1 scores eval

#use test data to get f1 stats

np.random.seed(42)
random.seed(42)
best_model2 = rs2.best_estimator_

#so we can see error of each epoch
#history2refine = best_model2refine.fit(x_train, y_train,
                    #validation_data = (x_test, y_test), class_weight = weight_class, verbose = 1)

y_pred2 = best_model2.predict(x_test)

#full confusion matrix printed below graph
cm2 = confusion_matrix(y_test, y_pred2)

print("Confusion matrix:")
print(cm2)

f1_2 = f1_score(y_test, y_pred2)
print(classification_report(y_test,y_pred2))

print("F1 Score: ", f1_2)


In [ ]:
print(classification_report(y_test,y_pred2))

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm2, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('2 Hidden Layer CM')

In [ ]:
# plt.figure()

# plt.plot(best_model2.history_["loss"], label="Training Loss")

# if "val_loss" in best_model2.history_:
#     plt.plot(best_model2.history_["val_loss"], label="Validation Loss")

# plt.xlabel("Epoch")
# plt.ylabel("Binary Cross-Entropy Loss 2 Hidden Layers")
# plt.title("Training vs Validation Loss")
# plt.legend()
# plt.show()

In [ ]:
#2 layers - refining search
#2 layers
#make MLP
def build_clf2(hidden1 = 32, hidden2 = 32, rate = 0.01, dropout_rate = 0, **kwargs):
    model2 = Sequential([
        Input(shape=(x.shape[1],)),
        Dense(hidden1, activation = 'relu'),
        Dense(hidden2, activation = 'relu'),
        Dropout(dropout_rate),
        Dense(1, activation = 'sigmoid')
    ])

#https://www.tensorflow.org/api_docs/python/tf/keras/Model
    
    model2.compile(optimizer=SGD(learning_rate = rate), loss = 'binary_crossentropy',
                  metrics = ['accuracy'])
    return model2

model2 = KerasClassifier(model=build_clf2, epochs = 10, batch_size = 128, verbose = 1)

#test diff params
params2 = {
    'model__hidden1':[32, 64, 128],
    'model__hidden2':[32, 64, 128],
    'model__rate':[0.0005, 0.001, 0.005],
    'epochs': [50, 100, 150],
    'model__dropout_rate': [0, 0.1, 0.3, 0.5]
}
#class sklearn.model_selection.RandomizedSearchCV(estimator, param_distributions, 
#*, n_iter=10, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, 
#pre_dispatch='2*n_jobs', random_state=None, error_score=nan, return_train_score=False)

#using random search 
random_search2 = RandomizedSearchCV(estimator = model2, param_distributions = params2,
                    n_iter = 50, scoring = 'f1', cv = 3, verbose = 1, 
                    random_state = 42)


In [ ]:
#fitting layer 2

#fit on training data
rs2 = random_search2.fit(x_train,y_train, class_weight=weight_class)


print("Best CV F1: ", rs2.best_score_)
print("Best Params: ", rs2.best_params_)

In [ ]:
#final best model + f1 scores eval

#use test data to get f1 stats

np.random.seed(42)
random.seed(42)
best_model2 = rs2.best_estimator_

#so we can see error of each epoch
#history2refine = best_model2refine.fit(x_train, y_train,
                    #validation_data = (x_test, y_test), class_weight = weight_class, verbose = 1)

y_pred2 = best_model2.predict(x_test)

#full confusion matrix printed below graph
cm2 = confusion_matrix(y_test, y_pred2)

print("Confusion matrix:")
print(cm2)

f1_2 = f1_score(y_test, y_pred2)
print(classification_report(y_test,y_pred2))

print("F1 Score: ", f1_2)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm2, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('2 Hidden Layer CM')

In [ ]:
y_prob2 = best_model2.predict_proba(x_val)

thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold2 = 0.5
y_pred2_best = (y_prob2[:,1] > 0.5).astype(int)
best_f1_2 = f1_score(y_val, y_pred2_best)

for i in thresholds :
    temp = (y_prob2[:,1] > i).astype(int)
    
    score = f1_score(y_val, temp)
    if score > best_f1_2:
        best_f1_2 = score
        best_threshold2 = i
        y_pred2_best = temp

cm2_best = confusion_matrix(y_val, y_pred2_best)

print("Confusion matrix:")
print(cm2_best)

print("F1 Score: ", best_f1_2)
print("Threshold: ", best_threshold2)

In [ ]:
y_prob2_test = best_model2.predict_proba(x_test)
y_pred2_test = (y_prob2_test[:,1] > best_threshold2).astype(int)

cm2_test = confusion_matrix(y_test, y_pred2_test)
f1_test2 = f1_score(y_test, y_pred2_test)

print("Confusion matrix:")
print(cm2_test)

print(classification_report(y_test,y_pred2_test))

print("F1 Score: ", f1_test2)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm2_test, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('2 Hidden Layer CM')